

## Architecture Overview

```
User Question
    ↓
[PLAN Node] → Analyze question
    ↓
[ROUTER Node with LLM] → Bedrock decides: Use RAG first? MCP? Web search?
    ↓
[TOOL EXECUTION] → Execute chosen tool with logging
    ↓
[HUMAN APPROVAL] → Show evidence, wait for "yes"
    ↓
[SYNTHESIZE] → Generate final answer
    ↓
Output
```

## Tool Selection Strategy:
1. **RAG (Retrieval)**: Search local documents first (fastest, private)
2. **MCP File System**: Access file system via MCP server (when file lookup needed)
3. **Web Search**: Fresh internet data (when RAG has no results)

**LLM decides** which tool to use based on the question content.

In [96]:
# Cell 1: Install and Import Required Libraries
import json
import os
from typing import TypedDict, Any, Dict, List
from langgraph.graph import StateGraph, START, END

print("✓ Imports successful")
print("✓ LangGraph is ready")

✓ Imports successful
✓ LangGraph is ready


In [97]:
import os
import boto3
from dotenv import load_dotenv
from langchain_aws import BedrockLLM

# Load .env file
load_dotenv("myenv.env")

# Read values
AWS_ACCESS_KEY_ID = os.getenv("AWS_ACCESS_KEY_ID")
AWS_SECRET_ACCESS_KEY = os.getenv("AWS_SECRET_ACCESS_KEY")
AWS_REGION = os.getenv("AWS_REGION", "us-east-1")

# Initialize Bedrock client with credentials
bedrock_runtime = boto3.client(
    service_name="bedrock-runtime",
    region_name=AWS_REGION,
    aws_access_key_id=AWS_ACCESS_KEY_ID,
    aws_secret_access_key=AWS_SECRET_ACCESS_KEY
)


class SimpleLLMRouter:
    """Uses Bedrock LLM for tool selection"""

    def __init__(self, client, model_id="amazon.nova-micro-v1:0"):
        self.client = client
        self.model_id = model_id

    def decide_tool(self, question: str):
        """
        Ask Bedrock LLM to decide which tool to use.
        """

        # Build request body in Nova schema
        body = {
            "messages": [
                {
                    "role": "user",
                    "content": [
                        {
                            "text": f"""
                            You are an AI router. The user asked: "{question}".
                            Decide which tool to use:
                            - 'rag' → Search local course documents first (Week 2 knowledge base).
                            - 'mcp' → Use MCPCalc for arithmetic questions or access files via MCP when the question mentions PDF, document, report, or file.
                            - 'web' → Search the internet for latest/current information.
                            If the question is a calculation, choose 'mcp' so the MCPCalc tool is used.
                            Respond ONLY with JSON: {{"tool_name": "...", "reason": "..."}}
                            """
                        }
                    ]
                }
            ],
            "inferenceConfig": {
                "maxTokens": 256,
                "temperature": 0.2
            }
        }

        # Call Bedrock
        response = self.client.invoke_model(
            modelId=self.model_id,
            body=json.dumps(body),
            contentType="application/json",
            accept="application/json",
        )

        # Parse Bedrock output
        output = json.loads(response["body"].read())
        text = output["output"]["message"]["content"][0]["text"]

        try:
            parsed = json.loads(text)
            return parsed.get("tool_name"), parsed.get("reason")
        except Exception:
            print("LLM output could not be parsed:", text)
            return None, None


# Instantiate the LLM router for tool selection
llm_router = SimpleLLMRouter(bedrock_runtime)


In [98]:
# Cell 3: RAG Document Store and Lookup
# Simulated local document database (Week 2 materials)

RAG_DOCUMENTS = {
    'machine learning': [
        'ML is about training models on data.',
        'Supervised learning uses labeled data.',
        'Week 2: Covered regression and classification.'
    ],
    'python': [
        'Python is a versatile programming language.',
        'Week 2 taught: data types, loops, functions.',
        'Python libraries: pandas, numpy, scikit-learn.'
    ],
    'ai': [
        'AI is the field of artificial intelligence.',
        'Week 2: Intro to AI concepts and applications.',
        'Large language models power modern AI.'
    ],
    'data science': [
        'Data science combines statistics and coding.',
        'Week 2: EDA, visualization, basic analysis.',
        'Data science workflow: collect → analyze → visualize.'
    ]
}

def rag_lookup(topic: str) -> Dict[str, Any]:
    """Search local RAG documents"""
    topic_lower = topic.lower()
    found_docs = []
    
    for key, docs in RAG_DOCUMENTS.items():
        if key in topic_lower or any(word in topic_lower for word in key.split()):
            found_docs.extend(docs)
    
    return {
        'status': 'found' if found_docs else 'not_found',
        'topic': topic,
        'documents': found_docs,
        'source': 'RAG (Local Knowledge Base - Week 2)'
    }

# Test RAG
test_result = rag_lookup('machine learning')
print(f"✓ RAG Lookup Test: {test_result['status']}")
print(f"  Found {len(test_result['documents'])} documents")

✓ RAG Lookup Test: found
  Found 3 documents


In [99]:
# MCPCalc client: attempts to call a prebuilt MCP calculator server ('mcp_server_calc')
import sys
import asyncio
import re

from typing import Any, Dict

def extract_math_expression(question: str) -> str:
    expression = question.strip()
    expression = re.sub(r'(?i)^(what is|calculate|compute|evaluate|solve|find|please)\s+', '', expression)
    expression = expression.rstrip('?. ')
    expression = re.sub(r'[^0-9+\-*/%^(). ]+', '', expression)
    expression = expression.replace('^', '**')
    return expression.strip()


def _run_async(coro):
    try:
        return asyncio.run(coro)
    except RuntimeError as e:
        if "asyncio.run() cannot be called from a running event loop" in str(e):
            try:
                import nest_asyncio
                nest_asyncio.apply()
            except Exception:
                pass
            loop = asyncio.get_event_loop()
            return loop.run_until_complete(coro)
        raise

def _mcp_calc_local_fallback(expr: str, server_error: str = None, import_error: str = None) -> Dict[str, Any]:
    try:
        import ast, operator as op
        operators = {
            ast.Add: op.add,
            ast.Sub: op.sub,
            ast.Mult: op.mul,
            ast.Div: op.truediv,
            ast.Pow: op.pow,
            ast.Mod: op.mod,
            ast.USub: op.neg,
        }
        def eval_expr(node):
            if isinstance(node, ast.Constant):
                return node.value
            if isinstance(node, ast.Num):
                return node.n
            if isinstance(node, ast.BinOp):
                return operators[type(node.op)](eval_expr(node.left), eval_expr(node.right))
            if isinstance(node, ast.UnaryOp):
                return operators[type(node.op)](eval_expr(node.operand))
            raise ValueError(f'Unsupported expression type: {type(node)}')
        tree = ast.parse(expr, mode='eval')
        val = eval_expr(tree.body)
        return {
            'status': 'found',
            'expression': expr,
            'result': str(val),
            'source': 'MCPCalc (local-fallback)',
            'server_error': server_error,
            'import_error': import_error,
        }
    except Exception as e:
        return {
            'status': 'error',
            'expression': expr,
            'error': str(e),
            'server_error': server_error,
            'import_error': import_error,
        }

try:
    import json
    from mcp import ClientSession, StdioServerParameters, stdio_client

    async def _call_mcpcalc_server(expr: str) -> str:
        server_params = StdioServerParameters(
            command=sys.executable,
            args=['-m', 'mcp_server_calc'],
            cwd='.',
        )
        async with stdio_client(server_params) as (read_s, write_s):
            async with ClientSession(read_s, write_s) as session:
                await session.initialize()
                res = await session.call_tool('calculate', {'expression': expr})
                if res.content:
                    texts = [getattr(b, 'text', '') for b in res.content if getattr(b, 'text', None)]
                    return '\n'.join(texts)
                if res.structuredContent is not None:
                    return json.dumps(res.structuredContent)
                return str(res)

    def mcp_calc_call(question: str) -> Dict[str, Any]:
        expr = extract_math_expression(question)
        if not expr:
            return {
                'status': 'error',
                'expression': question,
                'error': 'Could not parse math expression from question',
                'source': 'MCPCalc (MCP Server)',
            }
        try:
            result_text = _run_async(_call_mcpcalc_server(expr))
            return {
                'status': 'found',
                'expression': expr,
                'result': result_text,
                'source': 'MCPCalc (MCP Server)',
            }
        except Exception as e:
            return _mcp_calc_local_fallback(expr, server_error=str(e))
except Exception as e_import:
    def mcp_calc_call(question: str) -> Dict[str, Any]:
        expr = extract_math_expression(question)
        if not expr:
            return {
                'status': 'error',
                'expression': question,
                'error': 'Could not parse math expression from question',
                'source': 'MCPCalc (local-only)',
            }
        return _mcp_calc_local_fallback(expr, import_error=str(e_import))


In [100]:
# Cell 5: Web Search Tool with Real URLs (DuckDuckGo API)

from ddgs import DDGS

def web_search_tool(query: str) -> Dict[str, Any]:
    """
    Web Search Tool: Get latest information from the internet using DuckDuckGo
    Returns REAL URLs and snippets from actual search results
    """
    try:
        with DDGS() as ddgs:
            raw_results = list(ddgs.text(query, max_results=5))

        formatted_results = []
        for item in raw_results:
            title = item.get("title") or item.get("text") or "No title"
            snippet = item.get("body") or item.get("snippet") or item.get("text") or "No snippet"
            url = item.get("href") or item.get("link") or item.get("url") or "No URL"
            formatted_results.append({
                "title": title,
                "snippet": snippet,
                "url": url
            })

        if not formatted_results and raw_results:
            for item in raw_results:
                formatted_results.append({
                    "title": str(item),
                    "snippet": "",
                    "url": "No URL"
                })

        return {
            "status": "found" if formatted_results else "not_found",
            "query": query,
            "results": formatted_results,
            "source": "Web Search (DuckDuckGo - Real URLs)",
            "reason_to_use": "Latest/current information needed",
            "raw_result_count": len(raw_results)
        }
    except Exception as e:
        return {
            "status": "error",
            "query": query,
            "error": str(e),
            "source": "Web Search (DuckDuckGo)",
            "results": []
        }

# Test Web Search with real results
print("Testing web search with real DuckDuckGo API...")
test_web = web_search_tool("LangGraph AI agent")
print(f"✓ Web Search Test: {test_web['status']}")
print(f"  Retrieved {len(test_web.get('results', []))} real results from DuckDuckGo")
print(f"  Raw result count: {test_web.get('raw_result_count', 0)}")
if test_web.get('results'):
    print(f"  Example URLs:")
    for i, res in enumerate(test_web['results'][:2], 1):
        print(f"    [{i}] {res.get('title', 'No title')}")
        print(f"        URL: {res.get('url', 'No URL')}")


Testing web search with real DuckDuckGo API...
✓ Web Search Test: found
  Retrieved 5 real results from DuckDuckGo
  Raw result count: 5
  Example URLs:
    [1] Gmail AI Agent (LangGraph and Composio)
        URL: https://grokipedia.com/page/Gmail_AI_Agent_LangGraph_and_Composio
    [2] LangGraph: Agent Orchestration Framework for Reliable AI Agents
        URL: https://www.langchain.com/langgraph


In [101]:
# Cell 6: Logging Wrapper with Real URL Display

def log_tool_call(tool_name: str, inputs: Any) -> None:
    """Log tool call with inputs"""
    print(f"\n{'='*60}")
    print(f"🔧 TOOL CALL: {tool_name}")
    print(f"{'='*60}")
    print(f"Input Parameters:")
    print(json.dumps(inputs, default=str, indent=2))

def log_tool_result(tool_name: str, result: Any) -> None:
    """Log tool result with formatted URLs for web results"""
    print(f"\n{'─'*60}")
    print(f"✓ {tool_name} Result:")
    print(f"{'─'*60}")
    if isinstance(result, dict):
        print(f"Status: {result.get('status', 'unknown')}")
        print(f"Source: {result.get('source', 'unknown')}")
        print(f"\nContent Preview:")
        
        # Format RAG documents
        if result.get('documents'):
            for i, doc in enumerate(result['documents'][:3], 1):
                print(f"  [{i}] {doc[:100]}...")
        
        # Format WEB SEARCH results with clickable URLs
        elif result.get('results') and 'DuckDuckGo' in result.get('source', ''):
            print(f"Found {len(result['results'])} results:\n")
            for i, res in enumerate(result['results'][:3], 1):
                title = res.get('title', 'No title')
                url = res.get('url', 'No URL')
                snippet = res.get('snippet', 'No snippet')
                print(f"  [{i}] {title}")
                print(f"      🔗 URL: {url}")
                print(f"      📝 {snippet[:80]}...")
                print()
        
        # Format generic results
        elif result.get('results'):
            for i, res in enumerate(result['results'][:3], 1):
                title = res.get('title', 'No title')
                snippet = res.get('snippet', 'No snippet')
                print(f"  [{i}] {title}: {snippet[:70]}...")
        
        # Format MCP file content
        elif result.get('content'):
            print(f"  {result['content'][:150]}...")
    else:
        print(json.dumps(result, default=str, indent=2))

def log_error(tool_name: str, error: Exception) -> None:
    """Log errors"""
    print(f"\n❌ ERROR in {tool_name}:")
    print(f"   {str(error)}")

print("✓ Logging wrapper ready (with real URL formatting)")


✓ Logging wrapper ready (with real URL formatting)


In [102]:
# Cell 7: Define Graph State and Nodes

class ResearchState(TypedDict):
    question: str
    selected_tool: str  # Which tool LLM chose
    llm_reason: str    # Why LLM chose that tool
    evidence: Dict[str, Any]
    approved: bool
    final_answer: str

def plan_node(state: ResearchState) -> ResearchState:
    """Step 1: Analyze and prepare the question"""
    print(f"\n📋 PLAN NODE")
    print(f"Question: {state['question']}")
    return state

def router_node(state: ResearchState) -> ResearchState:
    """Step 2: LLM decides which tool to use"""
    print(f"\n🤖 ROUTER NODE (LLM Decision)")
    
    selected_tool, reason = llm_router.decide_tool(state['question'])
    
    if not selected_tool:
        selected_tool = 'rag'
        reason = 'Fallback to RAG because the router did not return a tool.'

    calc_keywords = ['calculate', 'compute', 'sum', 'add', 'subtract', 'multiply', 'divide', 'what is', 'evaluate', 'solve', '%']
    if any(k in state['question'].lower() for k in calc_keywords):
        selected_tool = 'mcp'
        reason = 'Calculation detected, route to MCPCalc via MCP.'

    print(f"LLM Decision:")
    print(f"  Tool: {selected_tool.upper()}")
    print(f"  Reason: {reason}")
    
    state['selected_tool'] = selected_tool
    state['llm_reason'] = reason
    return state

def research_node(state: ResearchState) -> ResearchState:
    """Step 3: Execute the tool selected by LLM"""
    print(f"\n🔍 RESEARCH NODE")
    
    tool = state['selected_tool']
    question = state['question']
    
    try:
        if tool == 'rag':
            log_tool_call('RAG_LOOKUP', {'query': question})
            result = rag_lookup(question)
            log_tool_result('RAG_LOOKUP', result)
            
        elif tool == 'mcp':
            calc_keywords = ['calculate', 'compute', 'sum', 'add', 'subtract', 'multiply', 'divide', 'what is', 'evaluate', 'solve', '%']
            if any(k in question.lower() for k in calc_keywords):
                expr = extract_math_expression(question)
                log_tool_call('MCPCalc', {'expression': expr})
                result = mcp_calc_call(expr)
                log_tool_result('MCPCalc', result)
            else:
                document_query_keywords = ['pdf', 'document', 'report', 'file', 'artificial intelligence']
                mcp_file = DOC_FILE_PATH if any(keyword in question.lower() for keyword in document_query_keywords) else sample_path
                log_tool_call('MCP_FILE_ACCESS', {'file': mcp_file})
                result = mcp_file_lookup(mcp_file)
                log_tool_result('MCP_FILE_ACCESS', result)
            
        elif tool == 'web':
            log_tool_call('WEB_SEARCH', {'query': question})
            result = web_search_tool(question)
            log_tool_result('WEB_SEARCH', result)
        
        state['evidence'] = result
        
    except Exception as e:
        log_error(f"{tool}_tool", e)
        state['evidence'] = {'status': 'error', 'error': str(e)}
    
    return state

def approval_node(state: ResearchState) -> ResearchState:
    """Step 4: Human in the loop - approve evidence"""
    print(f"\n👤 HUMAN APPROVAL NODE")
    print(f"\n{'='*60}")
    print("EVIDENCE SUMMARY:")
    print(f"{'='*60}")
    print(f"Tool Used: {state['selected_tool'].upper()}")
    print(f"Reason: {state['llm_reason']}")
    print(f"\nEvidence Details:")
    print(json.dumps(state['evidence'], indent=2))
    
    user_input = input(f"\nApprove this evidence? Type 'yes' to continue: ").strip().lower()
    state['approved'] = user_input == 'yes'
    
    return state

def synthesize_node(state: ResearchState) -> ResearchState:
    """Step 5: Generate final answer"""
    print(f"\n✨ SYNTHESIZE NODE")

    if not state['approved']:
        state['final_answer'] = "Research was not approved. Please try a different question."
        return state

    evidence = state['evidence']
    tool_used = state['selected_tool']

    if tool_used == 'rag' and evidence.get('status') == 'found':
        docs = evidence.get('documents', [])
        answer = f"From local knowledge base (RAG): {' '.join(docs[:2])}"
    elif tool_used == 'mcp' and evidence.get('status') == 'found':
        if 'result' in evidence:
            answer = f"From MCPCalc: {evidence['result']}"
        else:
            content = evidence.get('content', '')
            answer = f"From file system (MCP): {content[:150]}..."
    elif tool_used == 'web' and evidence.get('status') == 'found':
        snippets = [r['snippet'] for r in evidence.get('results', [])[:2]]
        answer = f"From web search: {' | '.join(snippets)}"
    else:
        answer = "No relevant information found."

    state['final_answer'] = answer
    return state

print("✓ All nodes defined successfully")


✓ All nodes defined successfully


In [104]:
# Cell 8: Build LangGraph Workflow

# Create the graph
graph = StateGraph(ResearchState)

# Add all nodes
graph.add_node('plan', plan_node)
graph.add_node('router', router_node)
graph.add_node('research', research_node)
graph.add_node('approval', approval_node)
graph.add_node('synthesize', synthesize_node)

# Define the flow
graph.set_entry_point('plan')
graph.add_edge('plan', 'router')          # Plan → Router (LLM decides tool)
graph.add_edge('router', 'research')      # Router → Research (Execute tool)
graph.add_edge('research', 'approval')    # Research → Approval (Human check)
graph.add_edge('approval', 'synthesize')  # Approval → Synthesize (Final answer)
graph.set_finish_point('synthesize')

# Validate the graph structure
graph.validate()

# Compile the graph for execution
compiled_graph = graph.compile()

print("✓ LangGraph workflow built successfully")
print("✓ Graph structure validated")
print("✓ Ready to execute research agent")

✓ LangGraph workflow built successfully
✓ Graph structure validated
✓ Ready to execute research agent


In [83]:
# Cell 9: Example 1 - Question about Local Knowledge

#print("\n" + "🚀 "*30)
print("EXAMPLE 1: Question about Python (in RAG)")
#print("🚀 "*30)

initial_state_1: ResearchState = {
    'question': 'What did we learn about Python in Week 2?',
    'selected_tool': '',
    'llm_reason': '',
    'evidence': {},
    'approved': False,
    'final_answer': ''
}

result_1 = compiled_graph.invoke(initial_state_1)

#print(f"\n{'='*60}")
print("FINAL ANSWER:")
#print(f"{'='*60}")
print(result_1['final_answer'])

EXAMPLE 1: Question about Python (in RAG)

📋 PLAN NODE
Question: What did we learn about Python in Week 2?

🤖 ROUTER NODE (LLM Decision)
LLM Decision:
  Tool: RAG
  Reason: The question is about specific course content from Week 2, which is best retrieved from the local course documents.

🔍 RESEARCH NODE

🔧 TOOL CALL: RAG_LOOKUP
Input Parameters:
{
  "query": "What did we learn about Python in Week 2?"
}

────────────────────────────────────────────────────────────
✓ RAG_LOOKUP Result:
────────────────────────────────────────────────────────────
Status: found
Source: RAG (Local Knowledge Base - Week 2)

Content Preview:
  [1] Python is a versatile programming language....
  [2] Week 2 taught: data types, loops, functions....
  [3] Python libraries: pandas, numpy, scikit-learn....

👤 HUMAN APPROVAL NODE

EVIDENCE SUMMARY:
Tool Used: RAG
Reason: The question is about specific course content from Week 2, which is best retrieved from the local course documents.

Evidence Details:
{
  "stat

In [84]:
# Cell 10: Example 2 - Question about Latest Information

#print("\n" + "🚀 "*30)
print("EXAMPLE 2: Question about Latest Trends (Web Search)")
#print("🚀 "*30)

initial_state_2: ResearchState = {
    'question': 'What are the latest AI trends today?',
    'selected_tool': '',
    'llm_reason': '',
    'evidence': {},
    'approved': False,
    'final_answer': ''
}

result_2 = compiled_graph.invoke(initial_state_2)

#print(f"\n{'='*60}")
print("FINAL ANSWER:")
#print(f"{'='*60}")
print(result_2['final_answer'])

EXAMPLE 2: Question about Latest Trends (Web Search)

📋 PLAN NODE
Question: What are the latest AI trends today?

🤖 ROUTER NODE (LLM Decision)
LLM Decision:
  Tool: WEB
  Reason: The question asks for the latest AI trends, which require current and up-to-date information from the internet.

🔍 RESEARCH NODE

🔧 TOOL CALL: WEB_SEARCH
Input Parameters:
{
  "query": "What are the latest AI trends today?"
}

────────────────────────────────────────────────────────────
✓ WEB_SEARCH Result:
────────────────────────────────────────────────────────────
Status: found
Source: Web Search (DuckDuckGo - Real URLs)

Content Preview:
Found 5 results:

  [1] What’s next in AI: 7 trends to watch in 2026
      🔗 URL: https://news.microsoft.com/source/features/ai/whats-next-in-ai-7-trends-to-watch-in-2026/
      📝 January 8, 2026 - In medicine, AI is helping close gaps in care. In software dev...

  [2] Latest Trends in Artificial Intelligence | element14 India
      🔗 URL: https://in.element14.com/latest-

In [105]:
# Cell 11: Example 3 - Question about Files (MCP)


print("EXAMPLE 3: evaluate 128*95+65%2")

initial_state_3: ResearchState = {
    'question': 'evaluate 128*95+65%2?',
    'selected_tool': '',
    'llm_reason': '',
    'evidence': {},
    'approved': False,
    'final_answer': ''
}

result_3 = compiled_graph.invoke(initial_state_3)

print("FINAL ANSWER:")
print(result_3['final_answer'])

EXAMPLE 3: evaluate 128*95+65%2

📋 PLAN NODE
Question: evaluate 128*95+65%2?

🤖 ROUTER NODE (LLM Decision)
LLM Decision:
  Tool: MCP
  Reason: Calculation detected, route to MCPCalc via MCP.

🔍 RESEARCH NODE

🔧 TOOL CALL: MCPCalc
Input Parameters:
{
  "expression": "128*95+65%2"
}

────────────────────────────────────────────────────────────
✓ MCPCalc Result:
────────────────────────────────────────────────────────────
Status: found
Source: MCPCalc (local-fallback)

Content Preview:

👤 HUMAN APPROVAL NODE

EVIDENCE SUMMARY:
Tool Used: MCP
Reason: Calculation detected, route to MCPCalc via MCP.

Evidence Details:
{
  "status": "found",
  "expression": "128*95+65%2",
  "result": "12161",
  "source": "MCPCalc (local-fallback)",
  "server_error": "unhandled errors in a TaskGroup (1 sub-exception)",
  "import_error": null
}

✨ SYNTHESIZE NODE
FINAL ANSWER:
From MCPCalc: 12161
